# ResNet50 v2/v3 — inference

This notebook does not modify checkpoint files. Run the first cell, select a non-empty file using `CONFIG["checkpoint_name"]`, then rerun all cells in order. Legacy v2 checkpoints are supported.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
from zipfile import ZipFile
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CONFIG = {
    "local": True,
    "checkpoint_name": None,  # Select a v2/v3 checkpoint from the list below.
    "use_subsample": False,
    "data_root_override": None,
    "test_manifest_override": None,
    "batch_size": 32,
    "export_tensorboard": True,
    "colab_root": Path("/content/drive/MyDrive/Colab Notebooks/AARHUS INTERNSHIP"),
    "results_root": "runs/inference_results",
}
LOCAL = CONFIG["local"]
CHECKPOINT_NAME = CONFIG["checkpoint_name"]
USE_SUBSAMPLE = CONFIG["use_subsample"]
DATA_ROOT_OVERRIDE = CONFIG["data_root_override"]
TEST_MANIFEST_OVERRIDE = CONFIG["test_manifest_override"]
BATCH_SIZE = CONFIG["batch_size"]
EXPORT_TENSORBOARD = CONFIG["export_tensorboard"]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if LOCAL:
    STORAGE_ROOT = PROJECT_ROOT / "output"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = CONFIG["colab_root"]
MODELS_DIR = STORAGE_ROOT / "models"
RESULTS_ROOT = PROJECT_ROOT / CONFIG["results_root"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Available checkpoints:")
for path in sorted(MODELS_DIR.glob("*.pth")):
    print(path.name, "|", path.stat().st_size, "byte", "EMPTY FILE" if path.stat().st_size == 0 else "")
    
# In Colab also copy src/ml_setup, or point SOURCE_ROOT to a clone's src.
import sys
SOURCE_ROOT = (PROJECT_ROOT / "src" if LOCAL else CONFIG["colab_root"] / "src")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from ml_setup.models import CustomResNet50
from ml_setup.helper_functions import test_step


## 1. Load and inspect the checkpoint

Set `CONFIG["checkpoint_name"]` in the previous cell. The version, scale, class labels, and normalization statistics are read from the checkpoint. Statistics are not recomputed on the test set.
Older files containing only model weights lack the required metadata and are rejected.


In [ ]:
if not CHECKPOINT_NAME:
    raise ValueError("CHECKPOINT_NAME must be set to one of the listed checkpoint files.")
CHECKPOINT_PATH = Path(CHECKPOINT_NAME)
if not CHECKPOINT_PATH.is_absolute():
    CHECKPOINT_PATH = MODELS_DIR / CHECKPOINT_PATH

checkpoint = torch.load(CHECKPOINT_PATH, 
                        map_location="cpu", 
                        weights_only=True)
required = { "representation", "scale", 
            "normalization", "idx_to_class",
            "model_state_dict", "completed_epochs", 
            "num_classes"}
if not isinstance(checkpoint, dict) or required - checkpoint.keys():
    raise ValueError("Checkpoint incompleto/legacy: richiesto il formato del notebook v2v3.")
representation_versions = {
                           "cumulative-rank-u16-v1": "v2", 
                           "cumulative-rank-u16-v2": "v2", 
                           "cumulative-rank-u16-v3": "v3"}
inferred_version = representation_versions.get(checkpoint["representation"])
VERSION, SCALE = checkpoint.get("version", inferred_version), checkpoint["scale"]

print("Checkpoint:", CHECKPOINT_PATH.resolve(), "| version:", VERSION, "| S:", SCALE)
print("Training manifest:", checkpoint.get("train_manifest"))
print("Test manifest:", checkpoint.get("test_manifest"))

REPRESENTATION = checkpoint["representation"]
if VERSION not in {"v2", "v3"} or VERSION != inferred_version:
    raise ValueError("Select a v2/v3 checkpoint; use v0_inference.ipynb for RGB")
if VERSION == "v3" and SCALE is not None:
    raise ValueError("Checkpoint v3 should have scale=None")
NUM_CLASSES = checkpoint["num_classes"]
idx_to_class = {int(k): v for k, v in checkpoint["idx_to_class"].items()}
if NUM_CLASSES != 10 or set(idx_to_class) != set(range(NUM_CLASSES)):
    raise ValueError("Expected a complete CIFAR-10 class mapping.")
CLASS_NAMES = [idx_to_class[i] for i in range(NUM_CLASSES)]
MEAN = float(checkpoint["normalization"]["mean"])
STD = float(checkpoint["normalization"]["std"])
if not np.isfinite([MEAN, STD]).all() or STD <= 0:
    raise ValueError("Normalization requires finite mean/std and std > 0.")
EPOCH = checkpoint["completed_epochs"]
history = checkpoint.get("history", [])
EXPERIMENT = f"v2_S{SCALE}" if VERSION == "v2" else "v3_adaptive"
print(f"{EXPERIMENT}; epochs={EPOCH}; mean={MEAN:.6f}; std={STD:.6f}")

## 2. Test set and model

Use extracted NPZ files, locally or on Drive. The default manifest is `output/<dataset>/test/manifest_test.json`, with NPZ files in the same folder.
Set `DATA_ROOT_OVERRIDE` for a different dataset root, or `TEST_MANIFEST_OVERRIDE` for a different manifest name/location (including subsamples). Extract `test.zip` before running this notebook; direct ZIP reading is not supported.
The loader checks class labels, representation, scale, dtype and shape. Inputs are converted to float32 and normalized as `(z - mean_train) / std_train`, without augmentation or division by 65535.


In [ ]:
class TestMaps(Dataset):
    def __init__(self, manifest_path):
        self.manifest_path = Path(manifest_path)
        self.folder = self.manifest_path.parent
        manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))
        mapping = {int(index): name for name, index in manifest["class_to_idx"].items()}
        if mapping != idx_to_class:
            raise ValueError("Test class mapping differs from the checkpoint.")
        self.samples = manifest["samples"]
        if not self.samples:
            raise ValueError("The test manifest contains no samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        source = self.folder / sample["file"]
        with ZipFile(source) as archive:
            representation = json.loads(archive.read("metadata.json"))["representation"]
        if representation_versions.get(representation.get("name")) != VERSION:
            raise ValueError(f"Rappresentazione errata: {sample['file']}")
        scale = representation.get("scale")
        if (VERSION == "v2" and scale != SCALE) or (VERSION == "v3" and
                (type(scale) is not int or not 0 <= scale <= 65535)):
            raise ValueError(f"Scala errata: {sample['file']}")
        with np.load(source, allow_pickle=False) as archive:
            z = archive["z_u16"]
        if z.dtype != np.uint16 or z.shape != (224, 224):
            raise ValueError("Atteso z_u16 uint16 [224,224]")
        x = torch.from_numpy(z.astype(np.float32)).unsqueeze(0)
        return (x - MEAN) / STD, sample["label"]

dataset_name = f"v2_cumulative_rank-s{SCALE}" if VERSION == "v2" else "v3_cumulative_rank_adaptive"
data_root = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else STORAGE_ROOT / dataset_name
if USE_SUBSAMPLE:
    data_root = data_root / "subsample"
TEST_MANIFEST = Path(TEST_MANIFEST_OVERRIDE) if TEST_MANIFEST_OVERRIDE else data_root / "test" / "manifest_test.json"
test_data = TestMaps(TEST_MANIFEST)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# Identica architettura/state_dict al CustomResNet50 del training; nessun download.
model = CustomResNet50(pretrained=False, num_classes=NUM_CLASSES, in_channels=1)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(DEVICE).eval()
print("Test samples:", len(test_data))

## 3. Evaluate the model

Run a forward pass over the test set. Loss is mean cross-entropy per sample. Predictions and probabilities are retained for later analysis and export. No training is performed.


In [ ]:
evaluation = test_step(model, tqdm(test_loader, desc="Test"), nn.CrossEntropyLoss(),
                       DEVICE, num_classes=NUM_CLASSES, return_predictions=True)
test_loss = evaluation["loss"]
confusion = np.asarray(evaluation["confusion_counts"], dtype=np.int64)
probabilities = evaluation["probabilities"]
targets = evaluation["targets"]
predictions = evaluation["predictions"]
evaluation_seconds = evaluation["evaluation_seconds"]
test_source = {"manifest": str(test_data.manifest_path)}
if not np.isfinite(test_loss):
    raise ValueError("Non-finite test loss")
print(f"Evaluation time: {evaluation_seconds:.2f}s (includes data loading and transfers)")


## 4. Metrics and confusion matrices

Rows represent true classes; columns represent predicted classes. The diagonal of the row-normalized matrix gives recall per class.
Precision, recall, and F1 are set to 0 when their denominators are zero. Rows with no samples are marked N/A in the normalized plot. Macro averages include all 10 classes.


In [ ]:
def metrics_from_confusion(cm):
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    tp = np.diag(cm).astype(float)
    def divide(a, b):
        return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)
    precision = divide(tp, predicted)
    recall = divide(tp, support)
    f1 = divide(2 * precision * recall, precision + recall)
    normalized = divide(cm.astype(float), support[:, None])
    per_class = pd.DataFrame({"class": CLASS_NAMES, "precision": precision,
                              "recall": recall, "f1": f1, "support": support})
    summary = {"num_samples": int(cm.sum()), "accuracy_pct": float(100 * tp.sum() / cm.sum()),
               "macro_precision": float(precision.mean()), "macro_recall": float(recall.mean()),
               "macro_f1": float(f1.mean()), "weighted_f1": float(np.dot(f1, support) / cm.sum())}
    return summary, per_class, normalized

summary, per_class, normalized = metrics_from_confusion(confusion)
summary.update({"test_loss": test_loss, "evaluation_seconds": evaluation_seconds, "epoch": EPOCH, "version": VERSION, "scale": SCALE})
display(pd.DataFrame([summary]))
display(per_class)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, values, title in zip(axes, [confusion, normalized], ["Conteggi", "Recall per riga"]):
    is_normalized = title != "Conteggi"
    im = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if is_normalized else None)
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES),
           xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
           xlabel="Classe predetta", ylabel="Classe vera", title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            label = (f"{values[i,j]:.1%}" if confusion[i].sum() else "N/A") if is_normalized else str(values[i,j])
            ax.text(j, i, label, ha="center", va="center", fontsize=8,
                    color="white" if values[i,j] > (values.max() / 2) else "black")
fig.suptitle(f"{EXPERIMENT} — epoca {EPOCH}")
fig.tight_layout()
plt.show()

## 5. Save results

Each execution creates a separate output folder; no checkpoint is overwritten.
`metrics.json` contains provenance information, results, and confusion matrices. CSV files support comparisons across runs.
Per-sample predictions and probabilities are saved in `predictions.npz`.


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_NAME = f"{CHECKPOINT_PATH.stem}_inference_{stamp}"
RUN_DIR = RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
report = {"schema": "resnet-v2v3-inference-v1", "checkpoint": str(CHECKPOINT_PATH),
          "mode": "inference", "created_utc": stamp,
          "test_source": test_source, "normalization": checkpoint["normalization"],
          "classes": CLASS_NAMES, "summary": summary, "zero_division": 0,
          "confusion_counts": confusion.tolist(), "confusion_proportions": normalized.tolist(),
          "per_class": per_class.to_dict(orient="records"),
          "device": str(DEVICE), "batch_size": BATCH_SIZE,
          "torch_version": str(torch.__version__)}
(RUN_DIR / "metrics.json").write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
pd.DataFrame([{**summary, "checkpoint": CHECKPOINT_PATH.name, "mode": "inference"}]).to_csv(RUN_DIR / "summary.csv", index=False)
per_class.to_csv(RUN_DIR / "per_class.csv", index=False)
for name, values in [("confusion_counts", confusion), ("confusion_proportions", normalized)]:
    pd.DataFrame(values, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RUN_DIR / f"{name}.csv", index_label="true_class")
pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)
fig.savefig(RUN_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
np.savez_compressed(RUN_DIR / "predictions.npz", targets=targets, predictions=predictions,
                    probabilities=probabilities, files=np.asarray([s["file"] for s in test_data.samples]),
                    class_names=np.asarray(CLASS_NAMES))
print("Risultati salvati in:", RUN_DIR)

## 6. Optional TensorBoard export

Set `EXPORT_TENSORBOARD=True` to export the training history and final test results. If the package is missing, install it in your environment with `%pip install tensorboard`.
The imported history preserves epoch numbers, **not the original event timestamps**. The current export logs training loss, accuracy, and learning rate; it does not export epoch durations.
Figures and scalars use shared tags to compare different checkpoints.


In [ ]:
if EXPORT_TENSORBOARD:
    from torch.utils.tensorboard import SummaryWriter
    LOG_ROOT = RESULTS_ROOT / "tensorboard"
    with SummaryWriter(log_dir=str(LOG_ROOT / RUN_NAME)) as writer:
        for entry in history:
            for key, tag in [("loss", "Loss/train"), ("accuracy", "Accuracy_pct/train"), ("lr", "Learning_rate")]:
                if key in entry:
                    writer.add_scalar(tag, entry[key], entry["epoch"])
        writer.add_scalar("Loss/test", test_loss, EPOCH)
        writer.add_scalar("Accuracy_pct/test", summary["accuracy_pct"], EPOCH)
        writer.add_scalar("F1_macro/test", summary["macro_f1"], EPOCH)
        for row in per_class.to_dict(orient="records"):
            for metric in ["precision", "recall", "f1"]:
                writer.add_scalar(f"Per_class/{row['class']}/{metric}", row[metric], EPOCH)
        writer.add_figure("Confusion_matrix/test", fig, EPOCH, close=False)
        writer.add_text("Configuration", json.dumps({"checkpoint": CHECKPOINT_PATH.name,
                        "version": VERSION, "scale": SCALE, "mode": "inference"}, indent=2))
    print("Log TensorBoard:", LOG_ROOT)
else:
    print("Esportazione TensorBoard disattivata; JSON/CSV e figure sono già salvati.")

To open TensorBoard in the notebook, run the following in a new cell:

```python
%load_ext tensorboard
%tensorboard --logdir "{RESULTS_ROOT / 'tensorboard'}"
```

Alternatively, run `tensorboard --logdir <path/to/inference_results/tensorboard>` in a terminal.
Repeat the notebook with other checkpoints to compare runs.
Compare results obtained on the same test samples. Repeated evaluations of a checkpoint are not independent training runs.
